# 09 · 약수터 물, 마셔도 될까? — 로지스틱 회귀와 재현율
교과서 **부록 221~226쪽** 선택 활동입니다. 데이터 탐구 이야기의 «약수터 음용 부적합» 실마리와 이어집니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

정답 `판정` 은 **1 = 적합(마셔도 됨) · 0 = 부적합**. 그런데 표기가 `Y`, `N`, `0(여시니아)` 처럼 섞여 있어 **전처리**가 먼저입니다.

## ① 불러오기

In [ ]:
주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
import pandas as pd

df = pd.read_csv(주소 + 'water_quality.csv')
df = df.drop('연번', axis=1)
df['판정'].value_counts()

## ② 변환 — 판정 표기 통일
💡 `Y` 는 적합(1), `N` 과 `0(여시니아)` 는 부적합(0). 글자 `'1'`, `'0'` 도 숫자로 바꿔야 합니다 → 마지막에 `astype(int)`.

In [ ]:
df['판정'] = df['판정'].replace({'Y': '1', 'N': '0', '0(여시니아)': '0'})
df['판정'] = df['판정'].astype(____)
print(df['판정'].value_counts())
print(df.isna().sum())
df = df.dropna()

**생각해 보기 ①** 적합과 부적합의 개수가 비슷한가요? 크게 차이 나면 «정확도»만 보면 어떤 함정이 있을까요? (Ⅱ-05 해석 함정 1번)

## ③ 나누기 · 학습
💡 두 갈래(적합/부적합) 판정 → `LogisticRegression`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

x = df.drop('판정', axis=1)
y = df['판정']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

model = ____(max_iter=1000)
model.fit(x_train, y_train)

## ④ 평가 — 정확도만 보면 안 되는 이유
💡 «부적합(0)»을 놓치면 위험합니다. 0에 대한 재현율을 보려면 `pos_label=0`.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

pred = model.predict(x_test)
print('정확도            :', round(accuracy_score(y_test, pred), 3))
print('기준선(전부 적합) :', round((y_test == 1).mean(), 3))
print('부적합 재현율     :', round(recall_score(y_test, pred, pos_label=____), 3))
print(confusion_matrix(y_test, pred, labels=[0, 1]))   # 행: 실제 0·1 / 열: 예측 0·1

## ⑤ 개선 실험 한 건 (수행평가 ② 요소 3 연습)
- **가설** 부적합이 적어서 모델이 «적합» 쪽으로 쏠린다.
- **조작** `class_weight='balanced'` 로 적은 쪽에 무게를 더 준다.
- **결과 · 해석** 아래를 실행하고 재현율과 정확도가 어떻게 바뀌는지 적어 보세요.

In [ ]:
m2 = LogisticRegression(max_iter=1000, class_weight='balanced').fit(x_train, y_train)
p2 = m2.predict(x_test)
print('정확도        :', round(accuracy_score(y_test, p2), 3))
print('부적합 재현율 :', round(recall_score(y_test, p2, pos_label=0), 3))

## ⑥ (선택) 새 값으로 판정해 보기

In [ ]:
항목 = list(x.columns)
값 = [10, 0, 0, 0.0, 2.0, 0.5]          # 일반세균, 총대장균군, 분원성대장균군, 암모니아성질소, 질산성질소, 과망간산칼륨소비량
p = m2.predict(pd.DataFrame([값], columns=항목))[0]
print('마셔도 돼요' if p == 1 else '마시면 안 돼요')

---
### 정답 예시 (막혔을 때만)
- ② `astype(int)` · ③ `LogisticRegression` · ④ `pos_label=0`
- 적합이 약 300, 부적합이 약 80으로 **불균형**입니다. 정확도가 기준선(약 0.79)과 비슷하면 «많은 쪽으로 찍기»와 다를 바 없습니다.
- `class_weight='balanced'` 는 적은 쪽(부적합)을 놓칠 때 벌을 더 줍니다. 이 데이터에서는 결과가 거의 그대로일 수 있습니다 — 그렇다면 **놓친 부적합 샘플**을 혼동 행렬로 찾아 어떤 수치였는지 살펴보는 것이 다음 개선입니다. 마시는 물에서는 정확도와 재현율 중 무엇이 더 중요할까요?